# Time-Series Trend & Rolling Metrics Analysis

This notebook demonstrates temporal analysis in Pandas, focusing on smoothing daily volatility, resampling across different time periods, tracking cumulative momentum, calculating period-over-period change rates, and identifying structural trends to inform strategic business decisions.

## Setup & Loading Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs("../output", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)

# Paths
RAW_DATA_PATH = "../data/raw/daily_revenue_data.csv"

# Load clean dataset
df = pd.read_csv(RAW_DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f"Loaded {len(df)} daily records from {RAW_DATA_PATH}.")
df.head()

## Task 1: Resample Data by Time Period

### Resampling vs. GroupBy
- **Resampling** is a time-aware operation that understands temporal relationships (frequencies like `W`, `M`, `Q`). It creates a complete calendar grid (meaning it will preserve empty periods if there are dates with zero activity) and can adjust timestamps (e.g. aligning to start or end of periods).
- **GroupBy** is a generic classification grouping. It groups strictly based on matching keys (e.g., string date, year/month keys) without any concept of calendar flow, gaps, or chronological order unless explicitly coded.

In [ ]:
df_ts = df.set_index('date')

# Weekly aggregation
weekly_revenue = df_ts['revenue'].resample('W').sum()
weekly_count = df_ts['orders'].resample('W').count()
weekly_avg = df_ts['revenue'].resample('W').mean()

weekly_metrics = pd.DataFrame({
    'weekly_revenue': weekly_revenue,
    'daily_records_count': weekly_count,
    'weekly_avg_daily_revenue': weekly_avg
})

# Monthly aggregation
monthly_revenue = df_ts['revenue'].resample('M').sum()
monthly_count = df_ts['orders'].resample('M').count()
monthly_avg = df_ts['revenue'].resample('M').mean()

monthly_metrics = pd.DataFrame({
    'monthly_revenue': monthly_revenue,
    'daily_records_count': monthly_count,
    'monthly_avg_daily_revenue': monthly_avg
})

print("=== WEEKLY METRICS (FIRST 5 WEEKS) ===")
print(weekly_metrics.head())
print("\n=== MONTHLY METRICS ===")
print(monthly_metrics)

# Compare results: find period of highest revenue
highest_weekly = weekly_metrics['weekly_revenue'].max()
highest_weekly_date = weekly_metrics['weekly_revenue'].idxmax().date()
highest_monthly = monthly_metrics['monthly_revenue'].max()
highest_monthly_date = monthly_metrics['monthly_revenue'].idxmax().strftime('%B %Y')

print(f"\nPeak Weekly Revenue: ${highest_weekly:,.2f} (Week ending {highest_weekly_date})")
print(f"Peak Monthly Revenue: ${highest_monthly:,.2f} ({highest_monthly_date})")

## Task 2: Compute Rolling Window Average

### Rolling Window Mechanics
A rolling window slides a fixed window of size $N$ across the timeline and performs an aggregation (like `mean()`) on the observations in that window. 
- **Window Size Selection**: 
  - **7-day window**: Captures and smooths out short-term weekly seasonality (e.g. weekend spikes vs weekday drops).
  - **30-day window**: Captures medium-term trends and smooths out monthly anomalies (e.g., paydays or promotional events).

In [ ]:
df['revenue_ma7'] = df['revenue'].rolling(window=7, min_periods=1).mean()
df['revenue_ma30'] = df['revenue'].rolling(window=30, min_periods=1).mean()

# Plot raw data alongside both rolling averages
plt.figure(figsize=(12, 6))
plt.plot(df['date'], df['revenue'], label='Raw Daily Revenue', color='#A2C2E8', alpha=0.5, linewidth=1)
plt.plot(df['date'], df['revenue_ma7'], label='7-Day Moving Average', color='#1F77B4', linewidth=2)
plt.plot(df['date'], df['revenue_ma30'], label='30-Day Moving Average', color='#FF7F0E', linewidth=2.5)

plt.title('Daily Revenue Trend: Raw vs. Rolling Averages', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Revenue ($)', fontsize=12)
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.legend(frameon=True, fontsize=10)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"${int(x/1000)}k"))
plt.tight_layout()
plt.savefig('../output/rolling_avg.png', dpi=300)
plt.show()

print("Rolling averages computed and plotted. The 30-day rolling line reveals a steady upward trend, while raw data shows severe daily fluctuations.")

## Task 3: Calculate Month-over-Month Percentage Change

### .pct_change() Explanation
- **`.pct_change()`** computes the percentage difference between the current value and the preceding value: $\frac{x_t - x_{t-1}}{x_{t-1}} \times 100$.
- **Negative values** indicate a contraction or decline compared to the previous period, while **positive values** show expansion or growth. A first row value is always `NaN` because it has no prior period to compare with.

In [ ]:
mom_change = monthly_metrics['monthly_revenue'].pct_change() * 100

# Show growth vs decline months
print("=== Month-over-Month Percentage Changes ===")
for idx, val in mom_change.items():
    month_name = idx.strftime('%B %Y')
    if pd.isna(val):
        print(f"{month_name}: N/A (Baseline Month)")
    else:
        print(f"{month_name}: {val:+.2f}%")

growth_months = mom_change[mom_change > 0]
decline_months = mom_change[mom_change < 0]

print(f"\nMonths with Positive Growth: {list(growth_months.index.strftime('%B %Y'))}")
print(f"Months with Negative Decline: {list(decline_months.index.strftime('%B %Y'))}")

## Task 4: Compute Cumulative Sum

### Running Total of Revenue
Cumulative sum tracks the total value accumulated up to each specific point in time, helping understand run-rate and total growth.

In [ ]:
df['cumulative_revenue'] = df['revenue'].cumsum()

plt.figure(figsize=(10, 5))
plt.plot(df['date'], df['cumulative_revenue'], color='#2CA02C', linewidth=2.5, label='Cumulative Revenue')
plt.fill_between(df['date'], df['cumulative_revenue'], color='#2CA02C', alpha=0.1)

plt.title('Cumulative Revenue Growth Over Time', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Accumulated Revenue ($)', fontsize=12)
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.legend(loc='upper left', frameon=True, fontsize=10)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig('../output/cumulative.png', dpi=300)
plt.show()

print(f"Total Revenue accumulated by end of period: ${df['cumulative_revenue'].iloc[-1]:,.2f}")

## Task 5: Identify Trend Pattern and Business Implications

In [ ]:
# Analyze rolling average trend over the last 30 days
recent_ma30 = df['revenue_ma30'].iloc[-30:]
trend_direction = 'up' if recent_ma30.iloc[-1] > recent_ma30.iloc[0] else 'down'
trend_magnitude = ((recent_ma30.iloc[-1] - recent_ma30.iloc[0]) / recent_ma30.iloc[0]) * 100

implication = "Accelerating growth - maintain current strategy" if trend_direction == 'up' else "Declining momentum - investigate causes"
revenue_volatility = df['revenue'].std()

analysis = f"""
TREND ANALYSIS:

Rolling Average Trend (last 30 days): {trend_direction.upper()}
Change over last 30 days: {trend_magnitude:+.1f}%

Month-over-month growth (latest month): {mom_change.iloc[-1]:+.1f}%

Business Implications:
- {implication}
- Revenue volatility (daily standard deviation): ${revenue_volatility:,.0f} (measure of noise)
"""
print(analysis)

## Bonus: Missing Data Handling Demo

### Handling Gaps in Time-Series
Time-series data often has missing records (due to system down-time, holidays, or missing logging). 
Steps to handle gaps:
1. **Reindex** to a complete `date_range` to expose the gaps as `NaN` values.
2. **Fill** the missing values:
   - **`ffill()` (Forward Fill)**: Propagates the last valid observation forward. Suitable for static values or when we assume status remains same.
   - **`bfill()` (Backward Fill)**: Propagates the next valid observation backward.
   - **`interpolate()`**: Estimates intermediate values. Ideal for continuous metrics like daily revenue to smooth out gaps linearly.

In [ ]:
# Drop 5% of rows randomly to create gaps
np.random.seed(42)
df_gaps = df.copy()
drop_indices = np.random.choice(df_gaps.index, size=int(len(df_gaps) * 0.05), replace=False)
df_gaps = df_gaps.drop(drop_indices).sort_values('date').reset_index(drop=True)
print(f"Created artificial gaps. Row count reduced from {len(df)} to {len(df_gaps)}.")

# Restore complete grid
full_grid = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
df_reindexed = df_gaps.set_index('date').reindex(full_grid)
print(f"Reindexed back to full grid. Number of NaNs in revenue: {df_reindexed['revenue'].isnull().sum()}")

# Compare filling techniques
df_reindexed['revenue_ffill'] = df_reindexed['revenue'].ffill()
df_reindexed['revenue_interpolated'] = df_reindexed['revenue'].interpolate(method='linear')

# Show a portion containing a filled gap
nan_rows = df_reindexed[df_reindexed['revenue'].isnull()].head()
print("\n=== Filled Gaps Comparison ===")
print(df_reindexed.loc[nan_rows.index[0] - pd.Timedelta(days=1) : nan_rows.index[0] + pd.Timedelta(days=1), 
                       ['revenue', 'revenue_ffill', 'revenue_interpolated']])